# Phase 26: Common Model Interface Design

**Goal:** We are about to code 6 completely different AI Models (Logistic Regression, Random Forest, XGBoost, LSTM, Standard Transformer, and a Lightweight Transformer).

If we don't enforce strict rules, each model will have different input/output shapes, making it impossible to compare them fairly. We will build a **Common Model Interface (CMI)**—a strict Python Abstract Base Class that forces every single model to play by the exact same rules!

In [1]:
import warnings
warnings.filterwarnings("ignore")
import sys  # type: ignore  # pylint: disable=import-error
!{sys.executable} -m pip install mlflow pydantic scikit-learn optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [optuna]2m1/2 [optuna]


In [2]:
import numpy as np  # type: ignore  # pylint: disable=import-error
import pandas as pd  # type: ignore  # pylint: disable=import-error
import time  # type: ignore  # pylint: disable=import-error
import os  # type: ignore  # pylint: disable=import-error
from abc import ABC, abstractmethod  # type: ignore  # pylint: disable=import-error
from enum import Enum  # type: ignore  # pylint: disable=import-error
from pathlib import Path  # type: ignore  # pylint: disable=import-error
from pydantic import BaseModel, Field  # type: ignore  # pylint: disable=import-error
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, classification_report  # type: ignore  # pylint: disable=import-error
from sklearn.exceptions import NotFittedError  # type: ignore  # pylint: disable=import-error


### Step 1: Abstract Model Interface (Subphase 26.1)
We use Python's `ABC` (Abstract Base Class). Any model that inherits from `XAIGuardModel` **must** implement `fit`, `predict`, `predict_proba`, `save`, and `load` exactly as defined here. If they don't, Python will literally crash the program before training even starts!

In [3]:
class ModelFamily(str, Enum):
    LOGISTIC_REGRESSION = "logistic_regression"
    RANDOM_FOREST = "random_forest"
    XGBOOST = "xgboost"
    LSTM = "lstm"
    TRANSFORMER = "transformer"
    LIGHTWEIGHT_TRANSFORMER = "lightweight_transformer"

class TrainingResult(BaseModel):
    model_family: str
    training_time_seconds: float
    best_params: dict
    validation_f1_macro: float
    mlflow_run_id: str

class XAIGuardModel(ABC):
    def __init__(self):
        self._is_fitted = False

    @property
    @abstractmethod
    def feature_names(self) -> list[str]:
        pass

    @property
    @abstractmethod
    def model_family(self) -> ModelFamily:
        pass

    @abstractmethod
    def fit(self, X_train: np.ndarray, y_train: np.ndarray, X_val: np.ndarray, y_val: np.ndarray) -> TrainingResult:
        pass

    @abstractmethod
    def predict(self, X: np.ndarray) -> np.ndarray:
        if not self._is_fitted:
            raise NotFittedError(f"This {self.__class__.__name__} is not fitted yet!")
        pass

    @abstractmethod
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        if not self._is_fitted:
            raise NotFittedError(f"This {self.__class__.__name__} is not fitted yet!")
        pass

    @abstractmethod
    def save(self, path: Path) -> None:
        pass

    @classmethod
    @abstractmethod
    def load(cls, path: Path) -> 'XAIGuardModel':
        pass

print("✅ Abstract Base Class Contract Established!")

✅ Abstract Base Class Contract Established!


### Step 2: Latency Profiler (Subphase 26.3)
We aren't just judging these AIs on accuracy. In Cybersecurity, an AI must be **FAST**. This LatencyProfiler uses `time.perf_counter_ns` (Nanosecond-level precision) to scientifically measure exactly how long it takes an AI to classify 1 packet.

In [4]:
class LatencyProfile(BaseModel):
    p50_ms: float
    p95_ms: float
    p99_ms: float
    throughput_events_per_sec: float

class LatencyProfiler:
    def __init__(self, warmup_runs: int = 1000, measurement_runs: int = 5000):
        self.warmup_runs = warmup_runs
        self.measurement_runs = measurement_runs
        
    def profile(self, model: XAIGuardModel, X_sample: np.ndarray) -> LatencyProfile:
        if len(X_sample) == 0:
            raise ValueError("Need at least 1 sample to profile latency!")
            
        single_event = X_sample[0:1]
        
        # 1. Warmup (Wake up CPU cache / PyTorch JIT)
        for _ in range(self.warmup_runs):
            _ = model.predict(single_event)
            
        # 2. Mathematical Measurement
        latencies_ns = []
        for _ in range(self.measurement_runs):
            start = time.perf_counter_ns()
            _ = model.predict(single_event)
            end = time.perf_counter_ns()
            latencies_ns.append(end - start)
            
        latencies_ms = np.array(latencies_ns) / 1_000_000.0
        
        p50 = np.percentile(latencies_ms, 50)
        p95 = np.percentile(latencies_ms, 95)
        p99 = np.percentile(latencies_ms, 99)
        
        avg_latency_s = (np.mean(latencies_ns) / 1_000_000_000.0)
        throughput = 1.0 / avg_latency_s if avg_latency_s > 0 else 0.0
        
        return LatencyProfile(
            p50_ms=float(p50),
            p95_ms=float(p95),
            p99_ms=float(p99),
            throughput_events_per_sec=float(throughput)
        )

print("✅ Latency Profiler Initialized!")

✅ Latency Profiler Initialized!


### Step 3: Three-Pillar Metrics Harness (Subphase 26.2)
This harness evaluates a model. Because every model inherits from `XAIGuardModel`, this single `evaluate()` function works flawlessly on XGBoost, LSTMs, and Transformers identically!

In [5]:
class ThreePillarMetrics(BaseModel):
    f1_macro: float
    f1_bruteforce: float
    latency_p99_ms: float
    shap_generation_successful: bool

class ModelEvaluationHarness:
    def __init__(self):
        self.profiler = LatencyProfiler(warmup_runs=50, measurement_runs=100) # Fast settings for test
        
    def evaluate(self, model: XAIGuardModel, X_test: np.ndarray, y_test: np.ndarray) -> ThreePillarMetrics:
        # Pillar 1: Accuracy
        y_pred = model.predict(X_test)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
        f1_brute = f1_score(y_test, y_pred, labels=[1], average='macro', zero_division=0)
        
        # Pillar 2: Explainability (Mocked until Phase 39)
        shap_success = True
        
        # Pillar 3: Latency
        profile = self.profiler.profile(model, X_test)
        
        return ThreePillarMetrics(
            f1_macro=float(f1_macro),
            f1_bruteforce=float(f1_brute),
            latency_p99_ms=profile.p99_ms,
            shap_generation_successful=shap_success
        )
        
print("✅ Universal Evaluation Harness Constructed!")

✅ Universal Evaluation Harness Constructed!


### Step 4: Interface Integration Test
Let's mathematically prove this works. We will build a tiny `DummyModel`, inherit our strict rules, train it on a fake dataset, and pass it to the universal harness!

In [6]:
from sklearn.linear_model import LogisticRegression  # type: ignore  # pylint: disable=import-error

# Notice how we MUST pass in XAIGuardModel here, forcing us to follow the rules!
class DummyModel(XAIGuardModel):
    def __init__(self):
        super().__init__()
        self.model = LogisticRegression()
        
    @property
    def feature_names(self):
        return ["feat_1", "feat_2"]
        
    @property
    def model_family(self):
        return ModelFamily.LOGISTIC_REGRESSION
        
    def fit(self, X_train, y_train, X_val, y_val):
        self.model.fit(X_train, y_train)
        self._is_fitted = True
        return TrainingResult(
            model_family=self.model_family,
            training_time_seconds=0.1,
            best_params={}, validation_f1_macro=0.99,
            mlflow_run_id="test-123"
        )
        
    def predict(self, X):
        super().predict(X) # Checks if fitted!
        return self.model.predict(X)
        
    def predict_proba(self, X):
        super().predict_proba(X)
        return self.model.predict_proba(X)
        
    def save(self, path): pass
    @classmethod
    def load(cls, path): return cls()

print("=== TESTING THE HARNESS ===")
# Fake Data
X = np.random.rand(10, 2)
y = np.array([0, 0, 1, 1, 2, 2, 0, 0, 1, 1])

model = DummyModel()
model.fit(X, y, X, y)

harness = ModelEvaluationHarness()
metrics = harness.evaluate(model, X, y)

print("\n✅ SUCCESS! The Universal Harness successfully evaluated the model:")
print(metrics.model_dump_json(indent=2))

=== TESTING THE HARNESS ===

✅ SUCCESS! The Universal Harness successfully evaluated the model:
{
  "f1_macro": 0.3666666666666667,
  "f1_bruteforce": 0.6,
  "latency_p99_ms": 0.22407992000000002,
  "shap_generation_successful": true
}


### Step 5: Serialisation Contract & Tests (Subphase 26.4)
When we deploy an AI to production, we load it from disk. If the loaded AI doesn't produce byte-identical predictions to the one we trained in the lab, we have a catastrophic ML engineering failure.

We simulate the `pytest` test suite here. Our DummyModel must pass all 6 rigorous Interface Contract Tests!

In [7]:
import os  # type: ignore  # pylint: disable=import-error
import joblib  # type: ignore  # pylint: disable=import-error
from pathlib import Path  # type: ignore  # pylint: disable=import-error

# Fix our DummyModel to actually support save/load for the test
class DummyModel(XAIGuardModel):
    def __init__(self):
        super().__init__()
        self.model = LogisticRegression()
        self._feature_names = ["feat_1", "feat_2"]
        
    @property
    def feature_names(self):
        return self._feature_names
        
    @property
    def model_family(self):
        return ModelFamily.LOGISTIC_REGRESSION
        
    def fit(self, X_train, y_train, X_val, y_val):
        self.model.fit(X_train, y_train)
        self._is_fitted = True
        return TrainingResult(model_family=self.model_family, training_time_seconds=0.1, best_params={}, validation_f1_macro=0.99, mlflow_run_id="test")
        
    def predict(self, X):
        super().predict(X)
        return self.model.predict(X)
        
    def predict_proba(self, X):
        super().predict_proba(X)
        return self.model.predict_proba(X)
        
    def save(self, path: Path) -> None:
        joblib.dump({"model": self.model, "features": self.feature_names}, path)
        
    @classmethod
    def load(cls, path: Path) -> XAIGuardModel:
        instance = cls()
        data = joblib.load(path)
        instance.model = data["model"]
        instance._feature_names = data["features"]
        instance._is_fitted = True
        return instance

print("=== RUNNING INTERFACE CONTRACT TESTS ===")
test_model = DummyModel()

# Test 1: NotFittedError
try:
    test_model.predict(X)
    print("❌ FAILED Test 1: Did not raise NotFittedError")
except NotFittedError:
    print("✅ PASSED Test 1: Raises NotFittedError before fit")

test_model.fit(X, y, X, y)
orig_preds = test_model.predict(X)

# Test 2: Predict shape and type
assert orig_preds.shape == (len(X),) and orig_preds.dtype in [np.int32, np.int64], "❌ FAILED Test 2"
print("✅ PASSED Test 2: Predict returns correct 1D integer shape")

# Test 3: Predict Proba sums to 1.0
probas = test_model.predict_proba(X)
assert np.allclose(np.sum(probas, axis=1), 1.0), "❌ FAILED Test 3"
print("✅ PASSED Test 3: Probabilities sum to exactly 1.0 per row")

# Test 4 & 5 & 6: Serialization Byte-Identicality & Metadata
test_path = Path("dummy_model.pkl")
test_model.save(test_path)
loaded_model = DummyModel.load(test_path)

loaded_preds = loaded_model.predict(X)
assert np.array_equal(orig_preds, loaded_preds), "❌ FAILED Test 4"
print("✅ PASSED Test 4: Loaded model predictions are strictly byte-identical to original")

assert loaded_model.feature_names == test_model.feature_names, "❌ FAILED Test 5"
print("✅ PASSED Test 5: Feature names preserved perfectly")

assert loaded_model.model_family == ModelFamily.LOGISTIC_REGRESSION, "❌ FAILED Test 6"
print("✅ PASSED Test 6: ModelFamily StrEnum matches correctly")

os.remove(test_path)
print("\n🎯 ALL SERIALISATION & INTERFACE TESTS PASSED!")

=== RUNNING INTERFACE CONTRACT TESTS ===
✅ PASSED Test 1: Raises NotFittedError before fit
✅ PASSED Test 2: Predict returns correct 1D integer shape
✅ PASSED Test 3: Probabilities sum to exactly 1.0 per row
✅ PASSED Test 4: Loaded model predictions are strictly byte-identical to original
✅ PASSED Test 5: Feature names preserved perfectly
✅ PASSED Test 6: ModelFamily StrEnum matches correctly

🎯 ALL SERIALISATION & INTERFACE TESTS PASSED!


### Step 6: Optuna Study Configuration (Subphase 26.5)
To tune our non-linear models (like XGBoost and Random Forest), we will use **Optuna's TPE Sampler**. 
This configuration ensures every trial uses Bayesian Probability to hunt for the best parameters, kills failing trials early (MedianPruner), and automatically logs each attempt to MLflow!

In [8]:
import optuna  # type: ignore  # pylint: disable=import-error
from optuna.samplers import TPESampler  # type: ignore  # pylint: disable=import-error
from optuna.pruners import MedianPruner  # type: ignore  # pylint: disable=import-error

def create_study(model_family: str, direction: str = "maximize", n_trials: int = 50, seed: int = 42) -> optuna.Study:
    # 1. TPE Sampler (Tree-structured Parzen Estimator)
    # Learns from past trials to guess where the best hyperparameters are!
    sampler = TPESampler(seed=seed, multivariate=True)
    
    # 2. Median Pruner
    # Kills unpromising trials after 10 epochs if they are performing worse than the median.
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=5)
    
    # 3. Create the Study
    study = optuna.create_study(
        study_name=f"{model_family}_optimization",
        direction=direction,
        sampler=sampler,
        pruner=pruner
    )
    return study

# Let's verify it builds correctly!
test_study = create_study("xgboost")
print(f"✅ Optuna Study '{test_study.study_name}' configured successfully with {type(test_study.sampler).__name__}!")

/Users/abdurrahman/.conda/envs/notebook-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-09-07 15:25:23,007] A new study created in memory with name: xgboost_optimization


✅ Optuna Study 'xgboost_optimization' configured successfully with TPESampler!
